# EQ-PROOF — Offline numeric validation + repair + signed proof

This notebook is a **working prototype** of the EQ-PROOF premise:

- Validate numeric outputs against a **JSON spec** of constraints  
  (bounds, equalities, simplex, monotonicity, sum≤cap)
- Optionally **repair** outputs to satisfy constraints (best-effort iterative projection)
- Emit a **signed proof artifact** (JSON + Markdown + PDF)
- Designed to run **offline** (no network required)

> Tip: Run top-to-bottom once, then use the UI section.


In [ ]:
# Core deps: all standard-library except PDF (reportlab) + crypto (cryptography)
import json, math, hashlib, datetime, base64
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import numpy as np

# PDF generation (installed in this environment)
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Ed25519 signing (cryptography is commonly available; fallback if missing)
try:
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey, Ed25519PublicKey
    from cryptography.hazmat.primitives import serialization
    CRYPTO_OK = True
except Exception as e:
    CRYPTO_OK = False
    _CRYPTO_ERR = str(e)

print("numpy:", np.__version__)
print("crypto:", "ok" if CRYPTO_OK else f"missing ({_CRYPTO_ERR})")


In [ ]:
# Optional UI (ipywidgets). If unavailable, the engine still works in CLI-style cells.
try:
    import ipywidgets as W
    from IPython.display import display, Markdown, clear_output
    WIDGETS_OK = True
except Exception as e:
    WIDGETS_OK = False
    _WIDGETS_ERR = str(e)

print("widgets:", "ok" if WIDGETS_OK else f"missing ({_WIDGETS_ERR})")


## 1) Spec format (JSON)

The spec is designed to be **human-writeable** and **machine-checkable**.

Each constraint has:
- `id`: stable identifier
- `type`: one of `bounds | equality | simplex | monotonic | sum_cap`
- `vars`: variable names (list)
- additional fields depending on type

Supported shapes:
- Values are stored in a dict: `{var_name: number or list-of-numbers}`

Examples are provided below and in the UI.


In [ ]:
DEMO_SPEC = {
  "meta": {
    "name": "demo_allocation_and_forecast",
    "version": "0.1",
    "tolerance": 1e-9
  },
  "constraints": [
    {"id": "w_bounds", "type": "bounds", "vars": ["w"], "min": 0.0, "max": 1.0},
    {"id": "w_simplex", "type": "simplex", "vars": ["w"]},

    {"id": "forecast_bounds", "type": "bounds", "vars": ["y"], "min": 0.0, "max": 1000.0},
    {"id": "forecast_monotone", "type": "monotonic", "vars": ["y"], "direction": "nondecreasing"},

    {"id": "budget_cap", "type": "sum_cap", "vars": ["costs"], "cap": 100.0}
  ]
}

DEMO_DATA = {
  # Allocation weights (should be nonnegative and sum to 1)
  "w": [0.7, 0.4, -0.2, 0.3],  # violates bounds and simplex

  # Forecast time series (should be nondecreasing, bounded)
  "y": [10, 8, 9, 15, 14, 16],  # violates monotonicity

  # Costs (should sum <= 100)
  "costs": [30, 40, 50]  # sums to 120
}

DEMO_SPEC, DEMO_DATA


## 2) Constraint engine (validate + repair)

### Validation output
- `ok`: pass/fail
- `violations`: list of structured violations (constraint id, variable, details)

### Repair strategy (prototype)
An **iterative projection loop**:
- Clip to bounds
- Enforce equality
- Project onto simplex
- Isotonic regression for monotonic series
- Scale down to satisfy sum cap

This is not a full optimizer, but it is deterministic, offline, and works well for many practical constraint stacks.


In [ ]:
def _hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def canonical_json(obj: Any) -> bytes:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")

def deep_copy_data(data: Dict[str, Any]) -> Dict[str, Any]:
    return json.loads(json.dumps(data))

def as_array(v: Any) -> np.ndarray:
    if isinstance(v, (list, tuple, np.ndarray)):
        return np.array(v, dtype=float)
    return np.array([float(v)], dtype=float)

def restore_shape(original: Any, arr: np.ndarray) -> Any:
    if isinstance(original, (list, tuple, np.ndarray)):
        return arr.tolist()
    return float(arr[0])

def project_simplex(v: np.ndarray) -> np.ndarray:
    # Euclidean projection onto simplex {x >= 0, sum(x)=1}
    # Standard algorithm: sort, find rho, threshold
    if v.size == 1:
        return np.array([1.0], dtype=float)
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u * np.arange(1, v.size + 1) > (cssv - 1))[0]
    if rho.size == 0:
        # fallback: all zeros -> put mass evenly
        return np.ones_like(v) / v.size
    rho = rho[-1]
    theta = (cssv[rho] - 1) / (rho + 1)
    w = np.maximum(v - theta, 0)
    # numeric cleanup
    s = w.sum()
    if s != 0:
        w = w / s
    return w

def isotonic_non_decreasing(y: np.ndarray) -> np.ndarray:
    # Pool Adjacent Violators (PAV) for nondecreasing isotonic regression
    y = y.astype(float).copy()
    n = len(y)
    if n <= 1:
        return y
    w = np.ones(n)
    i = 0
    while i < n - 1:
        if y[i] <= y[i+1] + 1e-15:
            i += 1
            continue
        # merge blocks backwards
        j = i
        while j >= 0 and y[j] > y[j+1] + 1e-15:
            tot_w = w[j] + w[j+1]
            avg = (y[j]*w[j] + y[j+1]*w[j+1]) / tot_w
            y[j] = avg
            w[j] = tot_w
            # delete j+1
            y = np.delete(y, j+1)
            w = np.delete(w, j+1)
            n -= 1
            j -= 1
        i = max(j+1, 0)
    # expand blocks back to original length
    # We lost structure by deletes; reconstruct by tracking block sizes during PAV:
    # Easier: implement with stacks.
    # We'll replace with stack-based implementation for correctness.
    return y

def isotonic_pav(y: np.ndarray, nondecreasing: bool = True) -> np.ndarray:
    y = y.astype(float)
    if y.size <= 1:
        return y.copy()
    if not nondecreasing:
        # for nonincreasing, flip sign
        return -isotonic_pav(-y, nondecreasing=True)
    # stack of (start, end, sum, weight)
    starts = []
    ends = []
    sums = []
    weights = []
    for i, val in enumerate(y):
        starts.append(i)
        ends.append(i)
        sums.append(val)
        weights.append(1.0)
        # merge while violation
        while len(sums) >= 2:
            avg_prev = sums[-2] / weights[-2]
            avg_last = sums[-1] / weights[-1]
            if avg_prev <= avg_last + 1e-15:
                break
            # merge last two blocks
            s = sums[-2] + sums[-1]
            w = weights[-2] + weights[-1]
            ends[-2] = ends[-1]
            sums[-2] = s
            weights[-2] = w
            # pop last
            starts.pop(); ends.pop(); sums.pop(); weights.pop()
    out = np.empty_like(y, dtype=float)
    for st, en, s, w in zip(starts, ends, sums, weights):
        out[st:en+1] = s / w
    return out

def validate(spec: Dict[str, Any], data: Dict[str, Any]) -> Dict[str, Any]:
    tol = float(spec.get("meta", {}).get("tolerance", 1e-9))
    violations = []
    for c in spec.get("constraints", []):
        ctype = c["type"]
        cid = c.get("id", "no_id")
        for var in c.get("vars", []):
            if var not in data:
                violations.append({"id": cid, "type": ctype, "var": var, "msg": "missing variable"})
                continue
            arr = as_array(data[var])

            if ctype == "bounds":
                mn = float(c.get("min", -np.inf))
                mx = float(c.get("max", np.inf))
                bad_lo = np.where(arr < mn - tol)[0].tolist()
                bad_hi = np.where(arr > mx + tol)[0].tolist()
                if bad_lo or bad_hi:
                    violations.append({
                        "id": cid, "type": ctype, "var": var,
                        "min": mn, "max": mx,
                        "bad_lo_idx": bad_lo, "bad_hi_idx": bad_hi,
                        "min_violation": float(np.min(arr - mn)) if bad_lo else 0.0,
                        "max_violation": float(np.max(arr - mx)) if bad_hi else 0.0
                    })

            elif ctype == "equality":
                target = float(c["value"])
                diffs = arr - target
                bad = np.where(np.abs(diffs) > tol)[0].tolist()
                if bad:
                    violations.append({
                        "id": cid, "type": ctype, "var": var,
                        "value": target, "bad_idx": bad,
                        "max_abs_diff": float(np.max(np.abs(diffs)))
                    })

            elif ctype == "simplex":
                if np.any(arr < -tol) or abs(arr.sum() - 1.0) > tol:
                    violations.append({
                        "id": cid, "type": ctype, "var": var,
                        "min": float(arr.min()), "sum": float(arr.sum())
                    })

            elif ctype == "monotonic":
                direction = c.get("direction", "nondecreasing")
                if direction not in ("nondecreasing", "nonincreasing"):
                    violations.append({"id": cid, "type": ctype, "var": var, "msg": f"bad direction {direction}"})
                    continue
                d = np.diff(arr)
                if direction == "nondecreasing":
                    bad = np.where(d < -tol)[0].tolist()
                else:
                    bad = np.where(d > tol)[0].tolist()
                if bad:
                    violations.append({
                        "id": cid, "type": ctype, "var": var,
                        "direction": direction, "bad_edges": bad
                    })

            elif ctype == "sum_cap":
                cap = float(c["cap"])
                s = float(arr.sum())
                if s > cap + tol:
                    violations.append({
                        "id": cid, "type": ctype, "var": var,
                        "cap": cap, "sum": s, "over": s - cap
                    })

            else:
                violations.append({"id": cid, "type": ctype, "var": var, "msg": f"unsupported type {ctype}"})
    return {"ok": len(violations) == 0, "violations": violations}

def repair(spec: Dict[str, Any], data: Dict[str, Any], max_iters: int = 50) -> Dict[str, Any]:
    tol = float(spec.get("meta", {}).get("tolerance", 1e-9))
    out = deep_copy_data(data)

    # Deterministic constraint order
    constraints = list(spec.get("constraints", []))

    for _ in range(max_iters):
        changed = False
        for c in constraints:
            ctype = c["type"]
            for var in c.get("vars", []):
                if var not in out:
                    continue
                original = out[var]
                arr = as_array(out[var])

                if ctype == "bounds":
                    mn = float(c.get("min", -np.inf))
                    mx = float(c.get("max", np.inf))
                    new = np.clip(arr, mn, mx)

                elif ctype == "equality":
                    target = float(c["value"])
                    new = np.full_like(arr, target, dtype=float)

                elif ctype == "simplex":
                    # also handles nonnegativity
                    new = project_simplex(arr)

                elif ctype == "monotonic":
                    direction = c.get("direction", "nondecreasing")
                    if direction == "nondecreasing":
                        new = isotonic_pav(arr, nondecreasing=True)
                    else:
                        new = isotonic_pav(arr, nondecreasing=False)

                elif ctype == "sum_cap":
                    cap = float(c["cap"])
                    s = float(arr.sum())
                    if s > cap + tol and s > 0:
                        new = arr * (cap / s)
                    else:
                        new = arr
                else:
                    new = arr

                if np.max(np.abs(new - arr)) > tol:
                    changed = True
                out[var] = restore_shape(original, new)

        if not changed:
            break

    report = validate(spec, out)
    return {"repaired": out, "ok": report["ok"], "violations": report["violations"], "iters": _+1}


In [ ]:
# Quick demo: validate + repair on the demo inputs
v0 = validate(DEMO_SPEC, DEMO_DATA)
r0 = repair(DEMO_SPEC, DEMO_DATA, max_iters=50)

v0, {"repair_ok": r0["ok"], "iters": r0["iters"], "violations_after": len(r0["violations"])}


## 3) Proof artifact (JSON + MD + PDF), signed offline

Artifact includes:
- spec hash, input hash, output hash
- validation summaries
- repair settings
- environment info
- signature (Ed25519) if crypto available


In [ ]:
import platform, sys

def env_fingerprint() -> Dict[str, Any]:
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    }

def sign_bytes_ed25519(payload: bytes, priv: Optional[Any] = None) -> Tuple[Optional[bytes], Optional[bytes], Optional[bytes]]:
    if not CRYPTO_OK:
        return None, None, None
    if priv is None:
        priv = Ed25519PrivateKey.generate()
    pub = priv.public_key()
    sig = priv.sign(payload)
    pub_bytes = pub.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw
    )
    priv_bytes = priv.private_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PrivateFormat.Raw,
        encryption_algorithm=serialization.NoEncryption()
    )
    return sig, pub_bytes, priv_bytes

def build_artifact(spec: Dict[str, Any], input_data: Dict[str, Any], output_data: Dict[str, Any],
                   validation_before: Dict[str, Any], validation_after: Dict[str, Any],
                   repair_meta: Dict[str, Any]) -> Dict[str, Any]:
    spec_bytes = canonical_json(spec)
    in_bytes = canonical_json(input_data)
    out_bytes = canonical_json(output_data)

    artifact = {
        "eqproof_version": "prototype-0.1",
        "spec_hash_sha256": _hash_bytes(spec_bytes),
        "input_hash_sha256": _hash_bytes(in_bytes),
        "output_hash_sha256": _hash_bytes(out_bytes),
        "validation_before": {
            "ok": validation_before["ok"],
            "violations": validation_before["violations"][:200],  # cap to keep artifacts small
            "violation_count": len(validation_before["violations"])
        },
        "validation_after": {
            "ok": validation_after["ok"],
            "violations": validation_after["violations"][:200],
            "violation_count": len(validation_after["violations"])
        },
        "repair": repair_meta,
        "environment": env_fingerprint(),
        "signature": None,
    }

    payload = canonical_json(artifact)
    sig, pub, priv = sign_bytes_ed25519(payload)
    if sig is not None:
        artifact["signature"] = {
            "alg": "ed25519",
            "sig_b64": base64.b64encode(sig).decode("ascii"),
            "pubkey_b64": base64.b64encode(pub).decode("ascii"),
        }
        # return priv separately to avoid embedding in proof (kept only locally by user)
        artifact["_local_private_key_b64"] = base64.b64encode(priv).decode("ascii")
    else:
        artifact["signature"] = {"alg": None, "note": "cryptography not available"}

    return artifact

def verify_artifact(artifact: Dict[str, Any]) -> bool:
    if not CRYPTO_OK:
        raise RuntimeError("cryptography not available; cannot verify signatures here.")
    siginfo = artifact.get("signature") or {}
    if siginfo.get("alg") != "ed25519":
        return False
    sig = base64.b64decode(siginfo["sig_b64"])
    pub = base64.b64decode(siginfo["pubkey_b64"])
    pubkey = Ed25519PublicKey.from_public_bytes(pub)

    # verify against the canonical form of the artifact with signature removed (exactly as signed)
    tmp = json.loads(json.dumps(artifact))
    tmp.pop("_local_private_key_b64", None)
    tmp["signature"] = None
    payload = canonical_json(tmp)
    pubkey.verify(sig, payload)
    return True


In [ ]:
# Build artifacts for the demo run
before = validate(DEMO_SPEC, DEMO_DATA)
rep = repair(DEMO_SPEC, DEMO_DATA, max_iters=50)
after = validate(DEMO_SPEC, rep["repaired"])

artifact = build_artifact(
    DEMO_SPEC, DEMO_DATA, rep["repaired"],
    before, after,
    repair_meta={"method": "iterative_projections", "max_iters": 50, "iters_used": rep["iters"]}
)

# Signature check (if crypto available)
sig_ok = None
if CRYPTO_OK and artifact.get("signature", {}).get("alg") == "ed25519":
    sig_ok = verify_artifact(artifact)

artifact["signature"], sig_ok


In [ ]:
# Export: JSON + Markdown + PDF
from pathlib import Path

OUTDIR = Path("eqproof_artifacts")
OUTDIR.mkdir(exist_ok=True)

def export_json(artifact: Dict[str, Any], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(artifact, f, indent=2, ensure_ascii=False)

def export_md(artifact: Dict[str, Any], path: Path) -> None:
    vb = artifact["validation_before"]
    va = artifact["validation_after"]
    lines = []
    lines.append(f"# EQ-PROOF Artifact — {artifact['eqproof_version']}")
    lines.append("")
    lines.append(f"- Spec hash (sha256): `{artifact['spec_hash_sha256']}`")
    lines.append(f"- Input hash (sha256): `{artifact['input_hash_sha256']}`")
    lines.append(f"- Output hash (sha256): `{artifact['output_hash_sha256']}`")
    lines.append("")
    lines.append("## Validation")
    lines.append(f"- Before: **{'PASS' if vb['ok'] else 'FAIL'}** ({vb['violation_count']} violations)")
    lines.append(f"- After: **{'PASS' if va['ok'] else 'FAIL'}** ({va['violation_count']} violations)")
    lines.append("")
    lines.append("## Repair")
    for k, v in artifact["repair"].items():
        lines.append(f"- {k}: `{v}`")
    lines.append("")
    lines.append("## Signature")
    sig = artifact.get("signature") or {}
    lines.append(f"- alg: `{sig.get('alg')}`")
    if sig.get("alg") == "ed25519":
        lines.append(f"- pubkey (b64): `{sig.get('pubkey_b64')[:24]}…`")
        lines.append(f"- sig (b64): `{sig.get('sig_b64')[:24]}…`")
    lines.append("")
    lines.append("## Environment")
    for k, v in artifact["environment"].items():
        lines.append(f"- {k}: `{v}`")
    lines.append("")
    lines.append("## Violations after (first 20)")
    for v in (va["violations"][:20] if va["violations"] else []):
        lines.append(f"- `{v.get('id')}` `{v.get('type')}` `{v.get('var')}` — {json.dumps(v, ensure_ascii=False)[:160]}…")
    if not va["violations"]:
        lines.append("- None ✅")

    path.write_text("\n".join(lines), encoding="utf-8")

def export_pdf(artifact: Dict[str, Any], path: Path) -> None:
    c = canvas.Canvas(str(path), pagesize=letter)
    width, height = letter
    x, y = 50, height - 50

    def line(txt, dy=14, size=10, bold=False):
        nonlocal y
        c.setFont("Helvetica-Bold" if bold else "Helvetica", size)
        c.drawString(x, y, txt[:110])
        y -= dy
        if y < 80:
            c.showPage()
            y = height - 50

    line("EQ-PROOF Proof Artifact", size=16, dy=22, bold=True)
    line(f"Version: {artifact['eqproof_version']}")
    line(f"Spec hash: {artifact['spec_hash_sha256']}")
    line(f"Input hash: {artifact['input_hash_sha256']}")
    line(f"Output hash: {artifact['output_hash_sha256']}")
    line("")
    vb, va = artifact["validation_before"], artifact["validation_after"]
    line("Validation", bold=True)
    line(f"Before: {'PASS' if vb['ok'] else 'FAIL'} ({vb['violation_count']} violations)")
    line(f"After:  {'PASS' if va['ok'] else 'FAIL'} ({va['violation_count']} violations)")
    line("")
    line("Repair", bold=True)
    for k, v in artifact["repair"].items():
        line(f"{k}: {v}")
    line("")
    sig = artifact.get("signature") or {}
    line("Signature", bold=True)
    line(f"alg: {sig.get('alg')}")
    if sig.get("alg") == "ed25519":
        line(f"pubkey(b64): {sig.get('pubkey_b64')[:48]}…")
        line(f"sig(b64):    {sig.get('sig_b64')[:48]}…")
    line("")
    line("Environment", bold=True)
    for k, v in artifact["environment"].items():
        line(f"{k}: {v}")
    line("")
    line("Violations after (first 10)", bold=True)
    if not va["violations"]:
        line("None")
    else:
        for v in va["violations"][:10]:
            line(f"{v.get('id')} {v.get('type')} {v.get('var')}")
    c.save()

json_path = OUTDIR / "artifact.json"
md_path = OUTDIR / "artifact.md"
pdf_path = OUTDIR / "artifact.pdf"

# Do not embed private key in the exported artifact
artifact_export = json.loads(json.dumps(artifact))
artifact_export.pop("_local_private_key_b64", None)

export_json(artifact_export, json_path)
export_md(artifact_export, md_path)
export_pdf(artifact_export, pdf_path)

(str(json_path), str(md_path), str(pdf_path))


## 4) UI — Spec + Data + Validate + Repair + Export

If widgets are available, this section gives a **single-screen** interactive app.

If widgets are not available, use the cells above as a CLI-style workflow.


In [ ]:
if not WIDGETS_OK:
    print("ipywidgets not available; skip UI. Engine still works.")
else:
    # --- Widgets
    spec_box = W.Textarea(
        value=json.dumps(DEMO_SPEC, indent=2),
        description="Spec JSON",
        layout=W.Layout(width="100%", height="260px")
    )
    data_box = W.Textarea(
        value=json.dumps(DEMO_DATA, indent=2),
        description="Data JSON",
        layout=W.Layout(width="100%", height="220px")
    )

    btn_validate = W.Button(description="Validate", button_style="primary")
    btn_repair = W.Button(description="Repair", button_style="warning")
    btn_export = W.Button(description="Export Artifact", button_style="success")

    max_iters = W.IntSlider(value=50, min=1, max=300, step=1, description="max iters", continuous_update=False)
    status = W.HTML(value="<b>Status:</b> idle")

    out_summary = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px", width="100%", height="220px"))
    out_violations = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px", width="100%", height="260px", overflow="auto"))
    out_export = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px", width="100%", height="130px"))

    state = {"spec": DEMO_SPEC, "data": DEMO_DATA, "repaired": None, "artifact": None}

    def parse_json_box(box: W.Textarea) -> Any:
        return json.loads(box.value)

    def render_validation(v: Dict[str, Any]):
        with out_summary:
            clear_output()
            print("PASS" if v["ok"] else "FAIL", "| violations:", len(v["violations"]))
        with out_violations:
            clear_output()
            if not v["violations"]:
                print("No violations ✅")
                return
            for i, viol in enumerate(v["violations"][:200], 1):
                print(f"{i:03d}. {viol.get('id')} | {viol.get('type')} | {viol.get('var')} | {json.dumps(viol, ensure_ascii=False)}")

    def do_validate(_=None):
        out_export.clear_output()
        try:
            spec = parse_json_box(spec_box)
            data = parse_json_box(data_box)
            state["spec"], state["data"] = spec, data
            v = validate(spec, data)
            render_validation(v)
            status.value = f"<b>Status:</b> validated — {'PASS' if v['ok'] else 'FAIL'}"
        except Exception as e:
            status.value = f"<b>Status:</b> error — {str(e)}"
            with out_summary:
                clear_output()
                print("Error:", e)

    def do_repair(_=None):
        out_export.clear_output()
        try:
            spec = parse_json_box(spec_box)
            data = parse_json_box(data_box)
            state["spec"], state["data"] = spec, data
            before = validate(spec, data)
            rep = repair(spec, data, max_iters=int(max_iters.value))
            after = validate(spec, rep["repaired"])
            state["repaired"] = rep["repaired"]
            render_validation(after)
            status.value = f"<b>Status:</b> repaired — {'PASS' if after['ok'] else 'FAIL'} (iters {rep['iters']})"
        except Exception as e:
            status.value = f"<b>Status:</b> error — {str(e)}"

    def do_export(_=None):
        try:
            spec = state["spec"]
            data = state["data"]
            repaired = state["repaired"]
            if repaired is None:
                raise ValueError("Run Repair first (need repaired output).")
            before = validate(spec, data)
            after = validate(spec, repaired)
            art = build_artifact(spec, data, repaired, before, after,
                                 repair_meta={"method": "iterative_projections", "max_iters": int(max_iters.value)})
            # export
            OUTDIR = Path("eqproof_artifacts_ui")
            OUTDIR.mkdir(exist_ok=True)
            artifact_export = json.loads(json.dumps(art))
            artifact_export.pop("_local_private_key_b64", None)
            export_json(artifact_export, OUTDIR / "artifact.json")
            export_md(artifact_export, OUTDIR / "artifact.md")
            export_pdf(artifact_export, OUTDIR / "artifact.pdf")

            with out_export:
                clear_output()
                print("Exported:")
                print(" -", OUTDIR / "artifact.json")
                print(" -", OUTDIR / "artifact.md")
                print(" -", OUTDIR / "artifact.pdf")
                if CRYPTO_OK and art.get("signature", {}).get("alg") == "ed25519":
                    print("Signature verified:", verify_artifact(art))
            status.value = "<b>Status:</b> exported"
        except Exception as e:
            with out_export:
                clear_output()
                print("Export error:", e)
            status.value = f"<b>Status:</b> export error — {str(e)}"

    btn_validate.on_click(do_validate)
    btn_repair.on_click(do_repair)
    btn_export.on_click(do_export)

    # Layout
    controls = W.HBox([btn_validate, btn_repair, btn_export, max_iters])
    left = W.VBox([W.HTML("<b>Spec</b>"), spec_box, W.HTML("<b>Data</b>"), data_box], layout=W.Layout(width="52%"))
    right = W.VBox([
        status,
        W.HTML("<b>Summary</b>"), out_summary,
        W.HTML("<b>Violations</b>"), out_violations,
        W.HTML("<b>Export</b>"), out_export
    ], layout=W.Layout(width="48%"))

    app = W.VBox([controls, W.HBox([left, right])])
    display(app)


## 5) Notes for production hardening

This prototype is intentionally compact. For a serious tool, you will likely want:
- A proper JSON schema + editor assistance
- Weighted objectives + hard/soft constraints
- Infeasibility diagnosis (small conflict sets)
- Batch / CI mode
- Better projection ordering (or a convex solver for coupled constraints)
